# Bronze Layer Notebook

In [0]:
# imports
from pyspark.sql.functions import explode, map_from_arrays, array, lit, col
from pyspark.sql.functions import substring, length, col
from pyspark.sql.functions import col, explode, when, trim
from pyspark.sql.types import StructType, StructField, MapType, StringType
from pyspark.sql.functions import col, collect_list, concat_ws, explode, from_json
from pyspark.sql.types import MapType, StringType, IntegerType


## Reading the transaction data

In [0]:
## JDBC connection variables
jdbc_url = (
    "jdbc:sqlserver://jrvssqlserver.database.windows.net:1433;"
    "database=financial_transactions_db;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

username = dbutils.secrets.get(scope="azure-sql-scope", key="sql-username")
password = dbutils.secrets.get(scope="azure-sql-scope", key="sql-password")

In [0]:
## Read transactions table from Azure SQL using JDBC
transactions_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.transactions_data")
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

display(transactions_df)

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,
7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,
7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,
7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,
7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,
7475333,2010-01-01 00:07:00,1807,165,$4.81,Swipe Transaction,20519,Bronx,NY,10464.0,5942,
7475334,2010-01-01 00:09:00,1556,2972,$77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,
7475335,2010-01-01 00:14:00,1684,2140,$26.46,Online Transaction,39021,ONLINE,,null,4784,
7475336,2010-01-01 00:21:00,335,5131,$261.58,Online Transaction,50292,ONLINE,,null,7801,
7475337,2010-01-01 00:21:00,351,1112,$10.74,Swipe Transaction,3864,Flushing,NY,11355.0,5813,


In [0]:
# writing to delta table
transactions_df.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.bronze.transactions_data"
)

## Reading the the User Data

In [0]:
# reading the users data and writing to delta
users_df = spark.read.option("header", True).option("inferSchema", True).csv(
    "abfss://bronze@jrvs.dfs.core.windows.net/raw/users_data.csv"
)

users_df.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.bronze.users_data"
)

## Reading he Cards Data

In [0]:
## Read cards table from Azure SQL using JDBC
cards_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.cards_data")
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

display(cards_df)

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No
4537,1746,Visa,Credit,4404898874682993,09/2003,736,YES,1,$27500,09/2003,2012,No
1278,1746,Visa,Debit,4001482973848631,07/2022,972,YES,2,$28508,02/2011,2011,No
3687,1746,Mastercard,Debit,5627220683410948,06/2022,48,YES,2,$9022,07/2003,2015,No
3465,1746,Mastercard,Debit (Prepaid),5711382187309326,11/2020,722,YES,2,$54,06/2010,2015,No
3754,1746,Mastercard,Debit (Prepaid),5766121508358701,02/2023,908,YES,1,$99,07/2006,2012,No


# Reading the the MCC Json data

In [0]:
# Read raw MCC JSON from the volume
mcc_raw_df = spark.read.option("multiline", "true").json(
    "/Volumes/pyspark_module/bronze/raw_json/mcc_codes.json"
)

display(mcc_raw_df)
mcc_raw_df.printSchema()

# Convert wide JSON object into key-value rows
mcc_df = (
    mcc_raw_df
    .select(
        explode(
            map_from_arrays(
                array(*[lit(c) for c in mcc_raw_df.columns]),
                array(*[col(c) for c in mcc_raw_df.columns])
            )
        ).alias("mcc_code", "description")
    )
)

display(mcc_df)
mcc_df.printSchema()

1711,3000,3001,3005,3006,3007,3008,3009,3058,3066,3075,3132,3144,3174,3256,3260,3359,3387,3389,3390,3393,3395,3405,3504,3509,3596,3640,3684,3722,3730,3771,3775,3780,4111,4112,4121,4131,4214,4411,4511,4722,4784,4814,4829,4899,4900,5045,5094,5192,5193,5211,5251,5261,5300,5310,5311,5411,5499,5533,5541,5621,5651,5655,5661,5712,5719,5722,5732,5733,5812,5813,5814,5815,5816,5912,5921,5932,5941,5942,5947,5970,5977,6300,7011,7210,7230,7276,7349,7393,7531,7538,7542,7549,7801,7802,7832,7922,7995,7996,8011,8021,8041,8043,8049,8062,8099,8111,8931,9402
"Heating, Plumbing, Air Conditioning Contractors",Steelworks,Steel Products Manufacturing,Miscellaneous Metal Fabrication,Miscellaneous Fabricated Metal Products,Coated and Laminated Products,Steel Drums and Barrels,Fabricated Structural Metal Products,"Tools, Parts, Supplies Manufacturing",Miscellaneous Metals,"Bolt, Nut, Screw, Rivet Manufacturing",Leather Goods,Floor Covering Stores,Upholstery and Drapery Stores,"Brick, Stone, and Related Materials",Pottery and Ceramics,Non-Ferrous Metal Foundries,"Electroplating, Plating, Polishing Services",Non-Precious Metal Services,Miscellaneous Metalwork,Heat Treating Metal Services,Welding Repair,Ironwork,Gardening Supplies,Industrial Equipment and Supplies,Miscellaneous Machinery and Parts Manufacturing,"Lighting, Fixtures, Electrical Supplies",Semiconductors and Related Devices,Passenger Railways,Ship Chandlers,Railroad Passenger Transport,Railroad Freight,Computer Network Services,Local and Suburban Commuter Transportation,Passenger Railways,Taxicabs and Limousines,Bus Lines,Motor Freight Carriers and Trucking,Cruise Lines,Airlines,Travel Agencies,Tolls and Bridge Fees,Telecommunication Services,Money Transfer,"Cable, Satellite, and Other Pay Television Services","Utilities - Electric, Gas, Water, Sanitary","Computers, Computer Peripheral Equipment",Precious Stones and Metals,"Books, Periodicals, Newspapers","Florists Supplies, Nursery Stock and Flowers",Lumber and Building Materials,Hardware Stores,Lawn and Garden Supply Stores,Wholesale Clubs,Discount Stores,Department Stores,"Grocery Stores, Supermarkets",Miscellaneous Food Stores,Automotive Parts and Accessories Stores,Service Stations,Women's Ready-To-Wear Stores,Family Clothing Stores,"Sports Apparel, Riding Apparel Stores",Shoe Stores,"Furniture, Home Furnishings, and Equipment Stores",Miscellaneous Home Furnishing Stores,Household Appliance Stores,Electronics Stores,Music Stores - Musical Instruments,Eating Places and Restaurants,Drinking Places (Alcoholic Beverages),Fast Food Restaurants,"Digital Goods - Media, Books, Apps",Digital Goods - Games,Drug Stores and Pharmacies,"Package Stores, Beer, Wine, Liquor",Antique Shops,Sporting Goods Stores,Book Stores,"Gift, Card, Novelty Stores","Artist Supply Stores, Craft Shops",Cosmetic Stores,"Insurance Sales, Underwriting","Lodging - Hotels, Motels, Resorts",Laundry Services,Beauty and Barber Shops,Tax Preparation Services,Cleaning and Maintenance Services,"Detective Agencies, Security Services",Automotive Body Repair Shops,Automotive Service Shops,Car Washes,Towing Services,"Athletic Fields, Commercial Sports","Recreational Sports, Clubs",Motion Picture Theaters,Theatrical Producers,"Betting (including Lottery Tickets, Casinos)","Amusement Parks, Carnivals, Circuses","Doctors, Physicians",Dentists and Orthodontists,Chiropractors,"Optometrists, Optical Goods and Eyeglasses",Podiatrists,Hospitals,Medical Services,Legal Services and Attorneys,"Accounting, Auditing, and Bookkeeping Services",Postal Services - Government Only


root
 |-- 1711: string (nullable = true)
 |-- 3000: string (nullable = true)
 |-- 3001: string (nullable = true)
 |-- 3005: string (nullable = true)
 |-- 3006: string (nullable = true)
 |-- 3007: string (nullable = true)
 |-- 3008: string (nullable = true)
 |-- 3009: string (nullable = true)
 |-- 3058: string (nullable = true)
 |-- 3066: string (nullable = true)
 |-- 3075: string (nullable = true)
 |-- 3132: string (nullable = true)
 |-- 3144: string (nullable = true)
 |-- 3174: string (nullable = true)
 |-- 3256: string (nullable = true)
 |-- 3260: string (nullable = true)
 |-- 3359: string (nullable = true)
 |-- 3387: string (nullable = true)
 |-- 3389: string (nullable = true)
 |-- 3390: string (nullable = true)
 |-- 3393: string (nullable = true)
 |-- 3395: string (nullable = true)
 |-- 3405: string (nullable = true)
 |-- 3504: string (nullable = true)
 |-- 3509: string (nullable = true)
 |-- 3596: string (nullable = true)
 |-- 3640: string (nullable = true)
 |-- 3684: string (null

mcc_code,description
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products
3007,Coated and Laminated Products
3008,Steel Drums and Barrels
3009,Fabricated Structural Metal Products
3058,"Tools, Parts, Supplies Manufacturing"
3066,Miscellaneous Metals


root
 |-- mcc_code: string (nullable = false)
 |-- description: string (nullable = true)



In [0]:
# writing the mcc data into json
mcc_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("pyspark_module.bronze.mcc_codes")

# Reading the fraud labels Json

In [0]:
# Read file as text lines
fraud_raw_text_df = spark.read.text(
    "/Volumes/pyspark_module/bronze/raw_json/train_fraud_labels.json"
)

# Combine all lines into one JSON string
fraud_json_string_df = fraud_raw_text_df.agg(
    concat_ws("", collect_list(col("value"))).alias("json_str")
)

display(fraud_json_string_df)

json_str


In [0]:

fraud_json_string_df.select(
    length(col("json_str")).alias("json_length"),
    substring(col("json_str"), 1, 1000).alias("json_preview")
).display()

json_length,json_preview
159083088,"{""target"": {""10649266"": ""No"", ""23410063"": ""No"", ""9316588"": ""No"", ""12478022"": ""No"", ""9558530"": ""No"", ""12532830"": ""No"", ""19526714"": ""No"", ""9906964"": ""No"", ""13224888"": ""No"", ""13749094"": ""No"", ""12303776"": ""No"", ""19480376"": ""No"", ""11716050"": ""No"", ""20025400"": ""No"", ""7661688"": ""No"", ""16662807"": ""No"", ""21419778"": ""No"", ""18011186"": ""No"", ""23289598"": ""No"", ""11644547"": ""No"", ""23235120"": ""No"", ""19748218"": ""No"", ""8720720"": ""No"", ""18335831"": ""No"", ""18936727"": ""No"", ""15223870"": ""No"", ""12370203"": ""No"", ""17126661"": ""No"", ""22270430"": ""No"", ""18790248"": ""No"", ""20143410"": ""No"", ""9497252"": ""No"", ""17619208"": ""No"", ""11052664"": ""No"", ""14670204"": ""No"", ""17681877"": ""No"", ""22485981"": ""No"", ""22332853"": ""No"", ""16628447"": ""No"", ""7766832"": ""No"", ""7614276"": ""No"", ""14069486"": ""No"", ""13755628"": ""No"", ""17306332"": ""No"", ""19822702"": ""No"", ""19118845"": ""No"", ""12799754"": ""No"", ""17368331"": ""No"", ""23652500"": ""No"", ""14024256"": ""No"", ""12296764"": ""No"", ""16044038"": ""No"", ""22500112"": ""No"", ""12343484"": ""No"", ""15796886"": ""No"", ""23508"


In [0]:
# Define schema for the outer wrapper
fraud_schema = StructType([
    StructField("target", MapType(StringType(), StringType()), True)
])

# Parse combined JSON string using the correct wrapper schema
fraud_parsed_df = fraud_json_string_df.select(
    from_json(col("json_str"), fraud_schema).alias("parsed")
)

display(fraud_parsed_df)
fraud_parsed_df.printSchema()

parsed


root
 |-- parsed: struct (nullable = true)
 |    |-- target: map (nullable = true)
 |    |    |-- key: string
 |    |    |-- value: string (valueContainsNull = true)



In [0]:
fraud_labels_df = fraud_parsed_df.select(
    explode(col("parsed.target")).alias("transaction_id", "is_fraud_raw")
)

display(fraud_labels_df)
fraud_labels_df.printSchema()

transaction_id,is_fraud_raw
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No
12532830,No
19526714,No
9906964,No
13224888,No
13749094,No


root
 |-- transaction_id: string (nullable = false)
 |-- is_fraud_raw: string (nullable = true)



In [0]:
#writing the fraud lables into delta table
fraud_labels_df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("pyspark_module.bronze.train_fraud_labels")